# 02c — Variabilidad entre semillas y test final de CISUTAC (bloque A)

El notebook 02 eligió las variantes **en valid y con una sola semilla** (`SEMILLA = 0`). Con valids de 43 (Holes) y 59 (Spots) fotos, una diferencia de pocas centésimas puede deberse sólo al azar del entrenamiento. El protocolo (`reports/protocolo_experimental.md`, sección 6) pide repetir la comparación final con tres semillas y reportar media y dispersión.

Este notebook **no elige nada**. Repite, con las configuraciones ya congeladas en el 02, las tres variantes que responden la pregunta de escala:

| ID | Variante | Configuración congelada |
|---|---|---|
| **A0** | Foto completa a 640 | igual que el 02 |
| **A1** | Foto completa a 1280 | igual que el 02 |
| **A3** | Tiles de 640, 20 % de solapamiento | Holes: 1 tile sano por tile con defecto; Spots: 0 (lo elegido en valid en el 02) |

Para cada semilla entrena, evalúa en valid (de donde sale el umbral de confianza) y evalúa **en test con ese umbral**. Es válido mirar test acá porque ninguna configuración se decide con estos números: sólo se describe cuánto varían.

- **La semilla cambia** la inicialización de la cabeza, el orden y los aumentos del entrenamiento y, en A3, qué tiles sanos se sortean.
- **Semilla 0 = corridas del 02.** Usan el mismo nombre que en el 02: si se agrega como *Input* el Output de esa corrida, se reutilizan los pesos y evaluaciones ya calculados (en Holes, esto da el **test de la A3 congelada**, que todavía no se había evaluado). Si no están, se reentrenan con la misma semilla.
- El criterio de elección del checkpoint (`best.pt` de Ultralytics) es el mismo del 02, para no mezclar esta pregunta con la propuesta del issue #5.

## 0. Parámetros

In [ ]:
DATASET = "holes"        # "holes" o "spots": una sesión de Kaggle por dataset
MODO_PRUEBA = True       # True: 1 época y pocas imágenes, sólo para verificar el pipeline
SEMILLAS = [0, 1, 2]     # 0 = la del notebook 02
EVALUAR_TEST = True      # test con el umbral de valid de cada corrida; nada se elige con estos números

# Configuración congelada en el notebook 02 (no cambiar acá)
MODELO = "yolo11n.pt"
EPOCAS_BASE, PACIENCIA_BASE = 100, 30
CONF_EVAL = 0.001
TILE, SOLAPAMIENTO, MIN_VISIBLE = 640, 0.2, 0.5
NEG_POR_POS = {"holes": 1.0, "spots": 0.0}[DATASET]
RESOLUCION_A1 = 1280
BATCH = {640: 16, 1280: 8}

if MODO_PRUEBA:
    EPOCAS_BASE, PACIENCIA_BASE = 1, 1
    FRACCION, LIMITE_EVAL, MAX_IMAGENES_TILES = 0.1, 3, 6
else:
    FRACCION, LIMITE_EVAL, MAX_IMAGENES_TILES = 1.0, None, None
SUFIJO = "_prueba" if MODO_PRUEBA else ""

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import cisutac_baseline as cb

pd.set_option("display.max_columns", 30)
print("Datos:", cb.DATA)
print("Resultados:", cb.RESULTS)
print("Dispositivo:", cb.pick_device())

if not (cb.derived_dir(DATASET) / "data.yaml").exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "prepare_cisutac_yolo.py")], check=True)
gt_train = cb.load_ground_truth(DATASET, "train")
n_completas = MAX_IMAGENES_TILES or len(gt_train["images"])


def nombre(variante, semilla):
    # Semilla 0 conserva el nombre del 02 para reutilizar sus corridas; las demás llevan sufijo.
    return f"{DATASET}_{variante}{'' if semilla == 0 else f'_s{semilla}'}{SUFIJO}"


def pesos(run_dir):
    return Path(run_dir) / "weights" / "best.pt"

## 1. Entrenamiento y evaluación por semilla

Mismo código que el 02 (`cb.train`, `cb.build_tile_dataset`, `cb.evaluate`). A3 usa el mismo presupuesto de imágenes vistas que A0. En test se reutiliza el umbral de confianza elegido en valid por esa misma corrida.

In [ ]:
def evaluar(run_name, weights, split, modo, umbral=None, **kw):
    detalle = f"completa{kw['imgsz']}" if modo == "completa" else f"tiles{kw['tile']}_o{round(100 * kw['overlap'])}"
    return cb.evaluate(f"{run_name}__{split}_{detalle}", DATASET, weights, split=split, mode=modo, conf=CONF_EVAL,
                       limit=LIMITE_EVAL, threshold=umbral, confirm_test=(split == "test"), **kw)


resultados = []
for semilla in SEMILLAS:
    print(f"===== Semilla {semilla} =====")
    corridas = {}
    for variante, imgsz in (("A0_completa640", 640), (f"A1_completa{RESOLUCION_A1}", RESOLUCION_A1)):
        run = cb.train(nombre(variante, semilla), cb.derived_dir(DATASET) / "data.yaml", imgsz=imgsz,
                       epochs=EPOCAS_BASE, patience=PACIENCIA_BASE, batch=BATCH[imgsz], seed=semilla,
                       model=MODELO, fraction=FRACCION)
        corridas[variante[:2]] = (run, "completa", {"imgsz": imgsz})

    yaml_t, stats_t = cb.build_tile_dataset(DATASET, tile=TILE, overlap=SOLAPAMIENTO, neg_per_pos=NEG_POR_POS,
                                            seed=semilla, min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES)
    epocas = cb.equivalent_epochs(EPOCAS_BASE, n_completas, stats_t["train"]["tiles_totales"],
                                  minimum=1 if MODO_PRUEBA else 10)
    paciencia = max(1, round(PACIENCIA_BASE * epocas / EPOCAS_BASE))
    run = cb.train(nombre(f"A3_tiles{TILE}_neg{NEG_POR_POS:g}", semilla), yaml_t, imgsz=TILE, epochs=epocas,
                   patience=paciencia, batch=BATCH[TILE], seed=semilla, model=MODELO)
    corridas["A3"] = (run, "tiles", {"tile": TILE, "overlap": SOLAPAMIENTO})

    for variante, (run, modo, kw) in corridas.items():
        m_valid = evaluar(run.name, pesos(run), "valid", modo, **kw)
        resultados.append({"variante": variante, "semilla": semilla, **m_valid})
        if EVALUAR_TEST:
            m_test = evaluar(run.name, pesos(run), "test", modo, umbral=m_valid["umbral_confianza"], **kw)
            resultados.append({"variante": variante, "semilla": semilla, **m_test})

tabla = pd.DataFrame(resultados)
display(cb.metrics_table(resultados))

## 2. Media y dispersión entre semillas

Desvío estándar muestral entre las semillas (con 3 semillas es una estimación gruesa; se muestran también el mínimo y el máximo). La pregunta es si la ventaja de A3 sobre A0 y A1 es mayor que lo que varía cada variante de una semilla a otra.

In [ ]:
METRICAS = ["mAP@0.5", "mAP@0.5:0.95", "AP_small", "precision", "recall", "f1", "segundos_por_imagen"]
resumen = (tabla.groupby(["split", "variante"])[METRICAS]
           .agg(["mean", "std", "min", "max"]).round(3))
salida = cb.RESULTS / "semillas"
salida.mkdir(parents=True, exist_ok=True)
resumen.to_csv(salida / f"{DATASET}_resumen{SUFIJO}.csv")
tabla.drop(columns=["encontrados_por_tamano", "ic95_recall", "ic95_f1"]).to_csv(
    salida / f"{DATASET}_por_semilla{SUFIJO}.csv", index=False)
resumen

**Diferencia pareada por semilla.** Para cada semilla, A3 menos A0 y A3 menos A1 en la misma métrica y el mismo split. Si la diferencia tiene el mismo signo en las tres semillas y es mayor que la dispersión de cada variante, la ventaja no es azar del entrenamiento.

In [ ]:
filas = []
for (split, semilla), g in tabla.groupby(["split", "semilla"]):
    v = g.set_index("variante")
    for rival in ("A0", "A1"):
        if {"A3", rival} <= set(v.index):
            filas.append({"split": split, "semilla": semilla, "comparación": f"A3 − {rival}",
                          **{m: v.loc["A3", m] - v.loc[rival, m] for m in ("mAP@0.5", "AP_small", "recall")}})
diferencias = pd.DataFrame(filas)
diferencias.to_csv(salida / f"{DATASET}_diferencias{SUFIJO}.csv", index=False)
display(diferencias.round(3))
display(diferencias.groupby(["split", "comparación"])[["mAP@0.5", "AP_small", "recall"]].agg(["mean", "min", "max"]).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, metrica in zip(axes, ("mAP@0.5", "AP_small")):
    for i, split in enumerate(sorted(tabla.split.unique(), reverse=True)):
        g = tabla[tabla.split == split]
        for j, variante in enumerate(["A0", "A1", "A3"]):
            valores = g[g.variante == variante][metrica]
            x = j + (i - 0.5) * 0.3
            ax.scatter([x] * len(valores), valores, color=f"C{i}", label=split if j == 0 else None, zorder=3)
            ax.hlines(valores.mean(), x - 0.1, x + 0.1, color=f"C{i}")
    ax.set_xticks(range(3), ["A0 640", f"A1 {RESOLUCION_A1}", "A3 tiles"])
    ax.set_title(f"{metrica} por semilla ({DATASET})")
    ax.grid(axis="y", alpha=0.3)
axes[0].legend()
plt.tight_layout()
plt.savefig(salida / f"{DATASET}_semillas{SUFIJO}.png", dpi=150)

**Defectos encontrados por tamaño en test**, sumando las tres semillas (cada defecto cuenta una vez por semilla).

In [ ]:
por_tamano = {}
for r in resultados:
    if r["split"] == "test":
        acum = por_tamano.setdefault(r["variante"], {})
        for rango, texto in r["encontrados_por_tamano"].items():
            hallados, total = map(int, texto.split("/"))
            h, t = acum.get(rango, (0, 0))
            acum[rango] = (h + hallados, t + total)
pd.DataFrame({v: {k: f"{h}/{t}" for k, (h, t) in d.items()} for v, d in por_tamano.items()}).T

## 3. Conclusiones

> Completar **después** de correr el notebook sin `MODO_PRUEBA`, con los números de las tablas.

- ¿La ventaja de A3 sobre A0 y A1 se mantiene en las tres semillas, en valid y en test?
- ¿Cuánto varía cada variante entre semillas, comparado con la diferencia entre variantes?
- Holes: ¿el test de la A3 (semilla 0) está dentro del rango de las otras semillas?